<a href="https://colab.research.google.com/github/Scurrra/ubpe/blob/master/examples/ubpe%20v0.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [UBPE](https://github.com/Scurrra/ubpe) - Universal Byte-Pair Encoding

[Link to Google Colab with computed cells](https://colab.research.google.com/drive/1QhQZlgggwtWByEWOwLcoWV37UgOA8VCI?usp=sharing)

## Classic BPE

While BPE itself is a gem, it can be generalized and improved.

First, BPE was develped for working with texts, but texts are not the only type of sequences, that can be tokenized.

Second, BPE is built to effectively merge pairs of tokens reqursively (still can be nicely done inplace without copying), but one pair at a time (as far as I know) during both training and inference. The procedure does not produce the best encoding (if TF-IDF is still a thing).

Third (optional), BPE substitution vocabulary is hard to analyze when needed.

## Universal BPE

`ubpe` package introduces several improvements: optimization for the classic algorithm and a brand-new BPE algorithm. As a bonus, it works with integer sequences out of the box!

The optimization is straightforward: why not create multiple tokens at each iteration? Anyway, we count all the pairs so we can select not one but many non-overlapping pairs. The optimization only creates one stylistic problem: a token with a higher number is no longer guaranteed to be less valuable than those with lower numbers. But this problem can be easily solved via rearranging.

BPE vocabulary is challenging to analyze: why were these tokens merged? Yes, they can be unwrapped, but what if we store the whole subsequences instead of substitution pairs? It will consume more storage, and it will require an entirely new encoding process...

With the new encoding algorithm, we can get many candidate encodings with top TF-IDF metrics. Furthermore, the lengths of the top candidates are smaller than the length of encoding achieved by the classic algorithm.

### Why would you need it?

As for now, with the current package version, the model can be trained on multiple encodings of the same data. Then, the most valuable encoding can be used during inference.

With some changes and improvements:
 - Full text search;
 - Typo correction;
 - Treating look-alike subsequences as the same.

### How to install

The [`ubpe`](https://pypi.org/project/ubpe/) package has two backends: Python native and C++ via Cython. The minimal supported Python version is 3.12. The package is compiled on Ubuntu 22.04 to support the Google Colab runtime.

The backends live in separate packages: [`ubpe-native`](https://pypi.org/project/ubpe-native/) and [`ubpe-cython`](https://pypi.org/project/ubpe-cython/). `ubpe` is a simple wrapper, backends are optional and must be installed via `ubpe[native]` or `ubpe[cython]`. Cython backend is always prioritized for import.

In [ ]:
!python --version

In [ ]:
!pip install "ubpe[cython]<0.3"

## Usage

In [ ]:
from datasets import load_dataset
from itertools import chain
from collections import Counter
from random import randint

from tqdm import tqdm
from IPython.display import display, HTML
import matplotlib
cmap = matplotlib.colormaps.get_cmap('PiYG')

import json, time

# timing function, as in Cython realization there is no tqdm
def timeit(fun):
    start = time.perf_counter()
    res = fun()
    end = time.perf_counter()
    elapsed = end - start
    print(f"\tElapsed time: {int(elapsed // 60)} m {elapsed % 60:.6f} s")
    return res

### Prepare dataset

In [ ]:
# select dataset
datasets = (
    # ~ 1.5 minutes to train each
    {"name": "ReactiveAI/TinyStories-mini-Interaction-SFT", "field": "answer"},
    # 4-5 minutes to train each
    {"name": "ReactiveAI/TinyStories-Interaction-SFT", "field": "answer"},
    # Exceeded RAM (dataset weight > 2GB)
    {"name": "roneneldan/TinyStories", "field": "text"},
)
dataset = datasets[0]

In [ ]:
# import corpus
corpus = [doc.lower() for doc in list(load_dataset(dataset["name"])["train"][dataset["field"]])]
corpus_test = [doc.lower() for doc in list(load_dataset(dataset["name"])["validation"][dataset["field"]])]

In [ ]:
# extract alphabet
alphabet = {token: id for id, (token, _) in enumerate(Counter(chain(*corpus)).most_common())}
print(f"Alphabet size: {len(alphabet)}")

In [ ]:
# select id to test
id = randint(0, len(corpus)-1)
id, corpus[id], len(corpus[id])

### Train tokenizers

> Attention! Type specification in class template matters when using the Cython backend. When using native backend, this specification is just a way to document code. But in `ubpe-cython` code `UBPEClassic[str]` is not actually a class template, but the element `libubpe.UbpeClassicChar` in the dict `UBPEClassic` with key `str` (Yeah, types themselves can be dictionary keys, i.e. are hashable!).

In [ ]:
import ubpe

In [ ]:
# maximal number of tokens in the vocabulary
n_tokens = 1024

# number of candidate tokens (maximal number of tokens) to add on each step
n_candidates = 100

In [ ]:
# create tokenizers
tokenizers = {
    "classic": ubpe.UBPEClassic[str](alphabet=alphabet, n_tokens=n_tokens),
    "novel": ubpe.UBPE[str](alphabet=alphabet, n_tokens=n_tokens)
}

#### Classic algorithm

In [ ]:
# fit classic
timeit(lambda: tokenizers["classic"].fit(corpus, n_candidates=n_candidates))

In [ ]:
# encoding example
encoded_sample = timeit(lambda: tokenizers["classic"].encode(corpus[id]))
len(encoded_sample[0][0]), encoded_sample[0][1]

In [ ]:
# decoding example
decoded_sample = timeit(lambda: tokenizers["classic"].decode(encoded_sample[0][0]))
len(decoded_sample), decoded_sample == corpus[id]

#### Novel algorithm

In [ ]:
# fit novel
timeit(lambda: tokenizers["novel"].fit(corpus, n_candidates=n_candidates))

In [ ]:
# encoding example
encoded_sample = timeit(lambda: tokenizers["novel"].encode(corpus[id]))
len(encoded_sample[0][0]), encoded_sample[0][1]

In [ ]:
# decoding example
decoded_sample = timeit(lambda: tokenizers["novel"].decode(encoded_sample[0][0]))
len(decoded_sample), decoded_sample == corpus[id]

### Dump and Load

The model can be dumped into a human-readable JSON string, stored, and than loaded from it.

In [ ]:
dump = tokenizers["classic"].dumps()
dump

In [ ]:
loaded = ubpe.UBPEClassic[str].loads(dump)

In [ ]:
# encoding example
encoded_sample = timeit(lambda: loaded.encode(corpus[id]))
len(encoded_sample[0][0]), encoded_sample[0][1]

In [ ]:
# decoding example
decoded_sample = timeit(lambda: loaded.decode(encoded_sample[0][0]))
len(decoded_sample), decoded_sample == corpus[id]

### Test on whole dataset

Takes a lot of time.

In [ ]:
# <= len(corpus_test)
max_i = 1000

In [ ]:
# check that encode-decode pair works properly all the time
i = 0
progress = tqdm(total=max_i, initial=i)
while i < max_i:
    encoded_sample = tokenizers["classic"].encode(corpus_test[i])
    decoded_sample = tokenizers["classic"].decode(encoded_sample[0][0])
    if decoded_sample == corpus_test[i]:
        i += 1
        progress.update()
    else:
        print(f"Error index = {i}")
        break
progress.close()

In [ ]:
# check that encode-decode pair works properly all the time
top_n = 3
i = 0
progress = tqdm(total=max_i, initial=i)
while i < max_i:
    encoded_sample = tokenizers["novel"].encode(corpus_test[i], top_n=top_n)
    decoded_sample = tokenizers["novel"].decode(encoded_sample[randint(0, top_n-1)][0])
    if decoded_sample == corpus_test[i]:
        i += 1
        progress.update()
    else:
        print(f"Error index = {i}")
        break
progress.close()

### Analysis

Let's analyze the difference in the results.

#### Prepare models

Unroll pairs in classic tokenizer and prepare visualization primitives.

In [ ]:
# @title
def generate_substrings_from_dump(dump: str, model_type: str = "novel"):
    dump = json.loads(dump)

    # unroll pairs into subsequences
    if model_type == "classic":
        cache = dict()
        for token in list(dump["mapper"].keys())[::-1]:
            pair = dump["mapper"][token]
            left = [pair[0]] if pair[0] < len(dump["alphabet"]) else cache[pair[0]]
            right = [pair[1]] if pair[1] < len(dump["alphabet"]) else cache[pair[1]]
            cache[int(token)] = left + right
        dump["mapper"] = {
            str(token): subseq
            for token, subseq in list(cache.items())[::-1]
        }

    max_weight = max(dump["weights"].values())
    vocab = {
        token : (
            char,
            0.0,
            f'''<span style="background-color: rgb{
                tuple(int(c * 255) for c in cmap(0.0)[:3]) + (0.72,)
            }; border: 1px solid black; border-radius: 3px; padding: 1px; font-size: 42px">{
                char
            }</span>'''
        )
        for char, token in dump["alphabet"].items()
    }
    for token_str, subseq in dump["mapper"].items():
        text = "".join(vocab[token][0] for token in subseq)
        vocab[int(token_str)] = (
            text,
            dump["weights"][token_str],
            f'''<span style="background-color: rgb{
                tuple(int(c * 255) for c in cmap(dump["weights"][token_str] / max_weight)[:3]) + (0.72,)
            }; border: 1px solid black; border-radius: 3px; padding: 1px; font-size: 42px" title={
                dump["weights"][token_str]
            }>{
                text
            }</span>'''
        )
    return vocab

In [ ]:
substrings_from_classic = generate_substrings_from_dump(tokenizers["classic"].dumps(), "classic")
substrings_from_novel = generate_substrings_from_dump(tokenizers["novel"].dumps(), "novel")

#### Visualization of text coverage with encoding tokens

> Token weight can be seen on hover (only non-alphabet tokens).

In [ ]:
# your text
text = ""

# text from validation corpus
# <= len(corpus_test)
id = 36
text = corpus_test[id]

In [ ]:
# @title
classic_encoding = tokenizers["classic"].encode(text)[0]
novel_encoding = tokenizers["novel"].encode(text)[0]

display(HTML(f"<h1>Text: </h1><h2>{text}</h2>"))
display(HTML(f"<h1>Encoding with classic algorithm ({len(classic_encoding[0])} tokens, TF-IDF = {classic_encoding[1]}):</h1>" + "".join(substrings_from_classic[token][2] for token in classic_encoding[0])))
display(HTML(f"<h1>Encoding with novel algorithm ({len(novel_encoding[0])} tokens, TF-IDF = {novel_encoding[1]}):</h1>" + "".join(substrings_from_novel[token][2] for token in novel_encoding[0])))